# Vv–Gamma安定微係数スイープ

`VvGammaChart.py` を用いて、基準OpenVSPモデルから垂直尾翼容積比 $V_v$ と翼端たわみ量 $w_{\mathrm{tip}}$ の組合せを生成し、VSPAERO安定微係数解析までを実行する。

このNotebookは形状生成と `.stab` 作成だけを担当する。旋回トリム、6自由度応答、統計集計、等値線描画は `plot_vv_gamma_chart.ipynb` で行う。


In [ ]:
from pathlib import Path
import os
import sys

import numpy as np

sys.path.append(os.path.join("../../../"))
from src.VvGammaChart import run_vv_wtip_stability_sweep


## 1. 入力条件

$V_v$ は11点、翼端たわみ量は21点とし、合計231ケースを計算する。主翼面積、翼幅、垂直尾翼モーメントアームはOpenVSPモデルから取得するため、`geometry_config` には重複値を持たせない。


In [ ]:
vv_values = np.linspace(0.001, 0.005, 5) + 0.0005
tip_deflections = 0.130207*(vv_values/0.001)-0.022240
print(vv_values)
print(tip_deflections)

In [ ]:
base_vsp3_path = Path("../../models/SampleGlider/SampleGlider.0G.vsp3")
output_dir = Path(".")

# vv_values = np.linspace(0.001, 0.006, 11)
# tip_deflections = np.linspace(0.0, 2.0, 21)

flight_condition = {
    "alpha_deg": 0.0,
    "mach": 0.0,
    "reynolds": 1.0e6,
}

geometry_config = {
    "xcg": 1.4,
    "wing_name": "WingGeom",
    "vtail_name": "VTailGeom",
    "n_span": 101,
    "vtail_area_scale_mode": "fixed_aspect_ratio",
}

print(f"case count = {len(vv_values) * len(tip_deflections)}")
print("vv_values       =", vv_values)
print("tip_deflections =", tip_deflections)


## 2. VSPAERO安定微係数スイープ

各ケースのディレクトリ、変形後 `.vsp3`、`.stab`、進捗情報を `output_dir` 以下へ保存する。


In [ ]:
sweep = run_vv_wtip_stability_sweep(
    base_vsp3_path,
    vv_values,
    tip_deflections,
    flight_condition,
    geometry_config,
    output_dir,
    validate_base_model=True,
    fixed_wake_flag=False,
    wake_num_iter=10,
    ncpu=8,
    output_csv_name="vv_wtip_stability_sweep.csv",
    verbose=2,
)

sweep.head()
